# Week 4 - Disney data cleaning, reshaping, and analysis

Combine the useful pandas examples and cleaning steps from the 2025 Week 3 and Week 4 notebooks.
The grouping, reshaping, and group transformation examples below extend that material for this session.

**Input:** `../data_raw/disney/disney_plus_shows.csv` (the historical Disney+ catalogue).
**Outputs:** `../output/data_clean/disney/` for cleaned data and annual movie files;
`../output/tables/disney/` for analysis tables.
This catalogue is different from the synthetic revenue workbook used in the pair activity.

**Analysis question:** among movies with at least 10,000 IMDb votes, how do ratings differ by
production-country group and release year? These are descriptive comparisons of a selected catalogue.
IMDb ratings are on a 0-10 scale; Metascores are on a 0-100 scale.

Run the notebook from this `scripts` folder, the Week 4 folder, or the repository root.
All input and output paths stay inside this year's Week 4 folder.

## 1. Imports and paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
root_candidates = [CWD, CWD.parent, CWD / "sessions" / "week_4"]
ROOT = next(
    (path for path in root_candidates
     if (path / "data_raw" / "disney" / "disney_plus_shows.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root, sessions/week_4, or its scripts folder.")

IN_RAW = ROOT / "data_raw" / "disney"
OUT = ROOT / "output"
OUT_CLEAN = OUT / "data_clean" / "disney"
OUT_TABLES = OUT / "tables" / "disney"
OUT_ANNUAL = OUT_CLEAN / "annual"
for folder in [OUT_CLEAN, OUT_TABLES, OUT_ANNUAL]:
    folder.mkdir(parents=True, exist_ok=True)

print("Input:", IN_RAW)
print("Cleaned data:", OUT_CLEAN)
print("Analysis tables:", OUT_TABLES)

Input: /Users/justinasgrigaitis/Documents/Github/ba-programming-2026/sessions/week_4/data_raw/disney
Cleaned data: /Users/justinasgrigaitis/Documents/Github/ba-programming-2026/sessions/week_4/output/data_clean/disney
Analysis tables: /Users/justinasgrigaitis/Documents/Github/ba-programming-2026/sessions/week_4/output/tables/disney


## 2. Import and inspect the catalogue

In [2]:
df_raw = pd.read_csv(IN_RAW / "disney_plus_shows.csv", thousands=",")
print("Rows and columns:", df_raw.shape)
display(df_raw.head())
df_raw.info()
display(df_raw.describe())
display(df_raw["type"].value_counts(dropna=False).rename("records"))

Rows and columns: (992, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 992 entries, 0 to 991
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   imdb_id      894 non-null    object 
 1   title        894 non-null    object 
 2   plot         866 non-null    object 
 3   type         894 non-null    object 
 4   rated        742 non-null    object 
 5   year         894 non-null    object 
 6   released_at  874 non-null    object 
 7   added_at     992 non-null    object 
 8   runtime      838 non-null    object 
 9   genre        885 non-null    object 
 10  director     689 non-null    object 
 11  writer       743 non-null    object 
 12  actors       870 non-null    object 
 13  language     856 non-null    object 
 14  country      869 non-null    object 
 15  awards       556 non-null    object 
 16  metascore    292 non-null    float64
 17  imdb_rating  879 non-null    float64
 18  imdb_votes   879 non-n

,imdb_id,title,plot,type,rated,year,released_at,added_at,runtime,genre,director,writer,actors,language,country,awards,metascore,imdb_rating,imdb_votes
0,tt0147800,10 Things I Hate About You,"A pretty, popular teenager can't go out on a d...",movie,PG-13,1999,31 Mar 1999,"November 12, 2019",97 min,"Comedy, Drama, Romance",Gil Junger,"Karen McCullah, Kirsten Smith","Heath Ledger, Julia Stiles, Joseph Gordon-Levi...","English, French",USA,2 wins & 13 nominations.,70.0,7.3,283945.0
1,tt7019028,101 Dalmatian Street,This series follows the lives of Delilah and D...,series,NaN,2018–,25 Mar 2019,"February 28, 2020",NaN,"Animation, Comedy, Family",NaN,NaN,"Josh Brener, Michaela Dietz, Bert Davis, Abiga...",English,"UK, USA, Canada",NaN,NaN,6.2,124.0
2,tt0115433,101 Dalmatians,An evil high-fashion designer plots to steal D...,movie,G,1996,27 Nov 1996,"November 12, 2019",103 min,"Adventure, Comedy, Crime, Family",Stephen Herek,"Dodie Smith (novel), John Hughes (screenplay)","Glenn Close, Jeff Daniels, Joely Richardson, J...","English, Spanish","USA, UK",Nominated for 1 Golden Globe. Another 3 wins &...,49.0,5.7,97785.0
3,tt0324941,101 Dalmatians 2: Patch's London Adventure,"Being one of 101 takes its toll on Patch, who ...",movie,G,2002,21 Jan 2003,"November 12, 2019",74 min,"Animation, Adventure, Comedy, Family, Musical","Jim Kammerud, Brian Smith","Jim Kammerud (story), Dan Root (story), Garret...","Barry Bostwick, Jason Alexander, Martin Short,...",English,USA,5 wins & 10 nominations.,NaN,5.8,7434.0
4,tt0211181,102 Dalmatians,Cruella DeVil gets out of prison and goes afte...,movie,G,2000,22 Nov 2000,"November 12, 2019",100 min,"Adventure, Comedy, Family",Kevin Lima,"Dodie Smith (novel), Kristen Buckley (story), ...","Glenn Close, Gérard Depardieu, Ioan Gruffudd, ...",English,"USA, UK",Nominated for 1 Oscar. Another 1 win & 7 nomin...,35.0,4.9,33444.0


,metascore,imdb_rating,imdb_votes
count,292.000000,879.000000,8.790000e+02
mean,62.061644,6.656428,6.254723e+04
std,15.776455,1.020352,1.721172e+05
min,19.000000,1.500000,5.000000e+00
25%,51.000000,6.100000,6.950000e+02
50%,61.000000,6.700000,3.952000e+03
75%,73.000000,7.400000,2.422300e+04
max,99.000000,9.700000,1.225316e+06


type
movie      680
series     191
NaN         98
episode     23
Name: records, dtype: int64

### Series, DataFrames, and sorting
One column selection returns a Series; a list of columns returns a DataFrame.
Sorting returns a new object unless `inplace=True` is supplied.

In [3]:
titles = df_raw["title"]
df_selected = df_raw[["title", "year", "imdb_rating"]]
print(type(titles), titles.dtype)
print(type(df_selected), df_selected.shape)
display(df_selected.dtypes)

titles_sorted = titles.sort_values(ascending=False)
display(titles_sorted.head())
display(df_selected.sort_values(["year", "title"]).head())

titles_copy = titles.copy()
titles_copy.sort_values(ascending=False, inplace=True)

<class 'pandas.core.series.Series'> object
<class 'pandas.core.frame.DataFrame'> (992, 3)


title           object
year            object
imdb_rating    float64
dtype: object

991                           Zootopia
990                          Zenon: Z3
989                  Zenon: The Zequel
988    Zenon: Girl of the 21st Century
987                             Zapped
Name: title, dtype: object

,title,year,imdb_rating
726,Steamboat Willie,1928,7.6
42,Babes in the Woods,1932,6.8
279,Flowers and Trees,1932,7.5
668,Santa's Workshop,1932,7.4
848,The Pied Piper,1933,6.7


### Create, rename, and drop columns
Convert both rating sources to the same 0-1 scale before combining them.
A simple arithmetic mean requires both ratings; the row-wise mean can use one available rating.
These examples use their own DataFrame so the raw input stays intact.

In [4]:
df_examples = df_raw.copy()
df_examples["metascore_pct"] = df_examples["metascore"] / 100
df_examples["imdb_pct"] = df_examples["imdb_rating"] / 10
df_examples["avg_score_both"] = (
    df_examples["metascore_pct"] + df_examples["imdb_pct"]
) / 2
df_examples["avg_score_available"] = df_examples[
    ["metascore_pct", "imdb_pct"]
].mean(axis=1, skipna=True)
display(df_examples[["title", "avg_score_both", "avg_score_available"]].head())

df_examples = df_examples.rename(columns={"avg_score_both": "avg_score_imdb_meta"})
df_examples = df_examples.drop(columns=["metascore_pct", "imdb_pct"])

raw_names = list(df_raw.columns)
renamed = [name + "_raw" for name in raw_names]
column_mapping = dict(zip(raw_names, renamed))
display(df_raw.rename(columns=column_mapping).head(2))

,title,avg_score_both,avg_score_available
0,10 Things I Hate About You,0.715,0.715
1,101 Dalmatian Street,NaN,0.620
2,101 Dalmatians,0.530,0.530
3,101 Dalmatians 2: Patch's London Adventure,NaN,0.580
4,102 Dalmatians,0.420,0.420


,imdb_id_raw,title_raw,plot_raw,type_raw,rated_raw,year_raw,released_at_raw,added_at_raw,runtime_raw,genre_raw,director_raw,writer_raw,actors_raw,language_raw,country_raw,awards_raw,metascore_raw,imdb_rating_raw,imdb_votes_raw
0,tt0147800,10 Things I Hate About You,"A pretty, popular teenager can't go out on a d...",movie,PG-13,1999,31 Mar 1999,"November 12, 2019",97 min,"Comedy, Drama, Romance",Gil Junger,"Karen McCullah, Kirsten Smith","Heath Ledger, Julia Stiles, Joseph Gordon-Levi...","English, French",USA,2 wins & 13 nominations.,70.0,7.3,283945.0
1,tt7019028,101 Dalmatian Street,This series follows the lives of Delilah and D...,series,NaN,2018–,25 Mar 2019,"February 28, 2020",NaN,"Animation, Comedy, Family",NaN,NaN,"Josh Brener, Michaela Dietz, Bert Davis, Abiga...",English,"UK, USA, Canada",NaN,NaN,6.2,124.0


## 3. Clean the variables used for analysis

In [5]:
columns_to_keep = [
    "imdb_id", "type", "title", "director", "language", "country", "year",
    "released_at", "runtime", "imdb_rating", "metascore", "imdb_votes", "rated",
]
df_clean = df_raw[columns_to_keep].copy()

### Numeric values and dates
For series with a year range, use the first year. Keep unknown years missing rather than assigning
the old sentinel value 9999. Parse runtimes in minutes and dates in day-month-year order.
`errors="raise"` exposes unexpected values instead of silently treating them as missing.

In [6]:
df_clean["year"] = pd.to_numeric(
    df_clean["year"].str[:4], errors="raise"
).astype("Int64")
df_clean["imdb_votes"] = pd.to_numeric(
    df_clean["imdb_votes"], errors="raise"
).astype("Int64")
df_clean["released_at"] = pd.to_datetime(
    df_clean["released_at"], format="%d %b %Y", errors="raise"
)
df_clean["runtime"] = pd.to_numeric(
    df_clean["runtime"].str.extract(r"(\d+)")[0], errors="raise"
).astype("Int64")
display(df_clean[["title", "year", "released_at", "runtime", "imdb_votes"]].head())

,title,year,released_at,runtime,imdb_votes
0,10 Things I Hate About You,1999,1999-03-31,97,283945
1,101 Dalmatian Street,2018,2019-03-25,<NA>,124
2,101 Dalmatians,1996,1996-11-27,103,97785
3,101 Dalmatians 2: Patch's London Adventure,2002,2003-01-21,74,7434
4,102 Dalmatians,2000,2000-11-22,100,33444


### Country and language groups
Split comma-separated values before testing group membership. The same function supports USA
production groups and English-language groups. Unknown metadata stays missing.

In [7]:
def classify_group(value, aliases, group_name):
    if pd.isna(value) or not value.strip():
        return pd.NA
    parts = {part.strip().lower() for part in value.split(",") if part.strip()}
    has_target = bool(parts & aliases)
    has_other = bool(parts - aliases)
    if has_target and has_other:
        return group_name + "_other"
    if has_target:
        return group_name
    return "other"

usa_aliases = {"usa", "u.s.a.", "united states", "united states of america"}
df_clean["country_group"] = df_clean["country"].apply(
    lambda value: classify_group(value, usa_aliases, "usa")
).astype("string")
df_clean["language_group"] = df_clean["language"].apply(
    lambda value: classify_group(value, {"english"}, "english")
).astype("string")
display(df_clean[["country", "country_group", "language", "language_group"]].head(10))

,country,country_group,language,language_group
0,USA,usa,"English, French",english_other
1,"UK, USA, Canada",usa_other,English,english
2,"USA, UK",usa_other,"English, Spanish",english_other
3,USA,usa,English,english
4,"USA, UK",usa_other,English,english
5,USA,usa,English,english
6,USA,usa,English,english
7,USA,usa,English,english
8,USA,usa,English,english
9,"USA, Australia, France, Canada",usa_other,English,english


### Derived scores and quality flags
Use conditions with `np.where` and `np.select`, then keep unavailable ratings distinct from low ratings.

In [8]:
df_clean["metascore_pct"] = df_clean["metascore"] / 100
df_clean["imdb_pct"] = df_clean["imdb_rating"] / 10
df_clean["avg_score_both"] = (
    df_clean["metascore_pct"] + df_clean["imdb_pct"]
) / 2
df_clean["avg_score_available"] = df_clean[["metascore_pct", "imdb_pct"]].mean(axis=1)
df_clean["quality_flag"] = np.where(df_clean["imdb_rating"] >= 7, "good", "bad")
conditions = [
    df_clean["imdb_rating"].ge(8.5),
    df_clean["imdb_rating"].between(7, 8.5, inclusive="left"),
    df_clean["imdb_rating"].between(5, 7, inclusive="left"),
]
df_clean["quality_group"] = np.select(conditions, ["top", "good", "mid"], default="bad")
missing_rating = df_clean["imdb_rating"].isna()
df_clean.loc[missing_rating, ["quality_flag", "quality_group"]] = pd.NA

### Missing values and duplicates
Inspect missing values before choosing what to remove. Missing IMDb IDs do not prove that rows
are duplicate movies. Check duplicated nonmissing IDs separately.

In [9]:
display(df_clean.isna().sum().rename("missing_values"))
duplicate_ids = df_clean["imdb_id"].notna() & df_clean["imdb_id"].duplicated(keep=False)
display(df_clean.loc[duplicate_ids, ["imdb_id", "title", "type"]])
display(df_clean[["year", "rated"]].drop_duplicates().head())

# This is an inspection view, rather than a rule to drop every incomplete record.
display(df_clean.dropna(subset=["imdb_rating"]).head())

imdb_id                 98
type                    98
title                   98
director               303
language               136
country                123
year                    98
released_at            118
runtime                154
imdb_rating            113
metascore              700
imdb_votes             113
rated                  250
country_group          123
language_group         136
metascore_pct          700
imdb_pct               113
avg_score_both         700
avg_score_available    113
quality_flag           113
quality_group          113
Name: missing_values, dtype: int64

,imdb_id,title,type


,year,rated
0,1999,PG-13
1,2018,NaN
2,1996,G
3,2002,G
4,2000,G


,imdb_id,type,title,director,language,country,year,released_at,runtime,imdb_rating,metascore,imdb_votes,rated,country_group,language_group,metascore_pct,imdb_pct,avg_score_both,avg_score_available,quality_flag,quality_group
0,tt0147800,movie,10 Things I Hate About You,Gil Junger,"English, French",USA,1999,1999-03-31,97,7.3,70.0,283945,PG-13,usa,english_other,0.70,0.73,0.715,0.715,good,good
1,tt7019028,series,101 Dalmatian Street,NaN,English,"UK, USA, Canada",2018,2019-03-25,<NA>,6.2,NaN,124,NaN,usa_other,english,NaN,0.62,NaN,0.620,bad,mid
2,tt0115433,movie,101 Dalmatians,Stephen Herek,"English, Spanish","USA, UK",1996,1996-11-27,103,5.7,49.0,97785,G,usa_other,english_other,0.49,0.57,0.530,0.530,bad,mid
3,tt0324941,movie,101 Dalmatians 2: Patch's London Adventure,"Jim Kammerud, Brian Smith",English,USA,2002,2003-01-21,74,5.8,NaN,7434,G,usa,english,NaN,0.58,NaN,0.580,bad,mid
4,tt0211181,movie,102 Dalmatians,Kevin Lima,English,"USA, UK",2000,2000-11-22,100,4.9,35.0,33444,G,usa_other,english,0.35,0.49,0.420,0.420,bad,bad


## 4. Select movies and define the analysis sample

In [10]:
df_movies = df_clean.loc[df_clean["type"] == "movie"].copy()
minimum_votes = 10_000
df_analysis = df_movies.loc[df_movies["imdb_votes"] >= minimum_votes].copy()
print("All movie records:", len(df_movies))
print("Movies with at least 10,000 IMDb votes:", len(df_analysis))

# Combine conditions with & and select columns with a list.
family_filter = (
    df_movies["imdb_rating"].between(6, 8, inclusive="both")
    & df_movies["rated"].isin(["PG", "PG-13"])
)
display(df_movies.loc[
    family_filter, ["title", "year", "rated", "imdb_rating"]
].sort_values("imdb_rating", ascending=False).head(10))

All movie records: 680
Movies with at least 10,000 IMDb votes: 280


,title,year,rated,imdb_rating
991,Zootopia,2016,PG,8.0
912,The Nightmare Before Christmas,1993,PG,8.0
493,Guardians of the Galaxy,2014,PG-13,8.0
498,The Avengers,2012,PG-13,8.0
811,The Incredibles,2004,PG,8.0
612,Pirates of the Caribbean: The Curse of the Bla...,2003,PG-13,8.0
418,Kitbull,2019,PG,8.0
495,Iron Man,2008,PG-13,7.9
500,Thor: Ragnarok,2017,PG-13,7.9
720,Star Wars: Episode VII - The Force Awakens,2015,PG-13,7.9


## 5. Reshape movie ratings: wide to long and back
The wide table has one row per movie and one column per rating source.
Convert IMDb ratings to a 0-100 scale so both columns use comparable units.
After `melt`, one row represents a movie and rating source; missing ratings remain visible.

In [11]:
df_ratings_wide = df_analysis[["imdb_id", "title", "year", "metascore"]].copy()
df_ratings_wide["imdb_score_100"] = df_analysis["imdb_rating"] * 10
rating_columns = ["imdb_score_100", "metascore"]
df_ratings_long = df_ratings_wide.melt(
    id_vars=["imdb_id", "title", "year"],
    value_vars=rating_columns,
    var_name="rating_source",
    value_name="score_100",
)
display(df_ratings_wide.head())
display(df_ratings_long.head())
print("Wide rows:", len(df_ratings_wide), "Long rows:", len(df_ratings_long))

Wide rows: 280 Long rows: 560


,imdb_id,title,year,metascore,imdb_score_100
0,tt0147800,10 Things I Hate About You,1999,70.0,73.0
2,tt0115433,101 Dalmatians,1996,49.0,57.0
4,tt0211181,102 Dalmatians,2000,35.0,49.0
6,tt0046672,"20,000 Leagues Under the Sea",1954,83.0,72.0
7,tt0120623,A Bug's Life,1998,77.0,72.0


,imdb_id,title,year,rating_source,score_100
0,tt0147800,10 Things I Hate About You,1999,imdb_score_100,73.0
1,tt0115433,101 Dalmatians,1996,imdb_score_100,57.0
2,tt0211181,102 Dalmatians,2000,imdb_score_100,49.0
3,tt0046672,"20,000 Leagues Under the Sea",1954,imdb_score_100,72.0
4,tt0120623,A Bug's Life,1998,imdb_score_100,72.0


### Pivot without aggregation
Each IMDb ID and rating source is unique, so `pivot` can restore the wide rating columns.
If those pairs were duplicated, decide why before choosing an aggregation rule.

In [12]:
df_ratings_back = df_ratings_long.pivot(
    index=["imdb_id", "title", "year"],
    columns="rating_source",
    values="score_100",
).reset_index()
df_ratings_back.columns.name = None
display(df_ratings_back.head())

,imdb_id,title,year,imdb_score_100,metascore
0,tt0029583,Snow White and the Seven Dwarfs,1937,76.0,95.0
1,tt0032455,Fantasia,1940,78.0,96.0
2,tt0032910,Pinocchio,1940,74.0,99.0
3,tt0034492,Bambi,1942,73.0,91.0
4,tt0038166,The Three Caballeros,1944,65.0,85.0


## 6. Summarize by production-country group and release year
After aggregation, one row represents a country group and year.
Count records separately from available ratings: an average ignores missing ratings.
`dropna=False` keeps unknown grouping values visible.

In [13]:
df_summary = df_analysis.groupby(
    ["country_group", "year"], as_index=False, dropna=False
).agg(
    movie_count=("imdb_id", "size"),
    rated_movie_count=("imdb_rating", "count"),
    mean_imdb_rating=("imdb_rating", "mean"),
    mean_runtime_minutes=("runtime", "mean"),
    total_imdb_votes=("imdb_votes", "sum"),
)
display(df_summary.head(12))

,country_group,year,movie_count,rated_movie_count,mean_imdb_rating,mean_runtime_minutes,total_imdb_votes
0,other,2004,1,1,6.9,98.0,24147
1,usa,1937,1,1,7.6,83.0,174379
2,usa,1940,2,2,7.6,106.5,209260
3,usa,1942,1,1,7.3,70.0,125824
4,usa,1944,1,1,6.5,71.0,11541
5,usa,1947,1,1,7.9,96.0,38715
6,usa,1949,1,1,7.0,68.0,11917
7,usa,1950,1,1,7.3,74.0,139378
8,usa,1951,1,1,7.4,75.0,123377
9,usa,1953,1,1,7.3,77.0,120707


### Pivot a summary and compare with `pivot_table`
The already-grouped table can use `pivot`. `pivot_table` starts from individual movies
and performs the aggregation itself. Empty group-year combinations stay missing, rather than zero.

In [14]:
df_summary_wide = df_summary.pivot(
    index="country_group", columns="year", values="mean_imdb_rating"
)
df_pivot_table = df_analysis.pivot_table(
    index="country_group", columns="year", values="imdb_rating",
    aggfunc="mean", dropna=False,
)
display(df_summary_wide)
display(df_pivot_table)

year,1937,1940,1942,1944,1947,1949,1950,1951,1953,1954,1955,1957,1959,1960,1961,1963,1964,1965,1967,1968,1970,1971,1973,1976,1977,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
country_group,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
other,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
usa,7.6,7.6,7.3,6.5,7.9,7.0,7.3,7.4,7.3,7.2,7.3,7.3,7.1,7.2,7.15,7.2,7.8,8.0,7.6,6.5,7.1,7.1,7.6,6.3,7.375,6.0,8.7,7.3,6.8,8.3,6.75,NaN,7.05,7.05,7.025,6.9,6.15,7.066667,6.55,6.733333,6.566667,7.2,5.620000,5.62,6.25,6.35,7.14,6.533333,7.000,6.4125,6.475,6.4625,6.9375,6.685714,6.45,5.95,6.88,6.64,6.95,7.225,7.475,7.3,7.6,7.183333,6.566667,7.433333,7.5
usa_other,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.6,NaN,7.2,NaN,NaN,NaN,6.55,NaN,NaN,NaN,6.3,6.90,NaN,7.70,6.400000,6.400000,6.8,6.433333,6.25,6.50,8.00,5.60,6.300000,5.525,6.7500,6.060,6.4500,5.0000,NaN,6.50,NaN,6.20,6.60,7.10,7.500,NaN,NaN,6.4,NaN,7.500000,6.700000,NaN


year,1937,1940,1942,1944,1947,1949,1950,1951,1953,1954,1955,1957,1959,1960,1961,1963,1964,1965,1967,1968,1970,1971,1973,1976,1977,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
country_group,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
other,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
usa,7.6,7.6,7.3,6.5,7.9,7.0,7.3,7.4,7.3,7.2,7.3,7.3,7.1,7.2,7.15,7.2,7.8,8.0,7.6,6.5,7.1,7.1,7.6,6.3,7.375,6.0,8.7,7.3,6.8,8.3,6.75,NaN,7.05,7.05,7.025,6.9,6.15,7.066667,6.55,6.733333,6.566667,7.2,5.620000,5.62,6.25,6.35,7.14,6.533333,7.000,6.4125,6.475,6.4625,6.9375,6.685714,6.45,5.95,6.88,6.64,6.95,7.225,7.475,7.3,7.6,7.183333,6.566667,7.433333,7.5
usa_other,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.6,NaN,7.2,NaN,NaN,NaN,6.55,NaN,NaN,NaN,6.3,6.90,NaN,7.70,6.400000,6.400000,6.8,6.433333,6.25,6.50,8.00,5.60,6.300000,5.525,6.7500,6.060,6.4500,5.0000,NaN,6.50,NaN,6.20,6.60,7.10,7.500,NaN,NaN,6.4,NaN,7.500000,6.700000,NaN


### Group transformation preserves the movie rows
Add the mean rating for each movie's country group, then calculate its distance from that mean.
The result still has one row per movie, unlike the summary table.

In [15]:
df_transformed = df_analysis.copy()
df_transformed["country_mean_rating"] = df_transformed.groupby(
    "country_group", dropna=False
)["imdb_rating"].transform("mean")
df_transformed["rating_minus_country_mean"] = (
    df_transformed["imdb_rating"] - df_transformed["country_mean_rating"]
)
display(df_transformed[[
    "title", "country_group", "imdb_rating", "country_mean_rating", "rating_minus_country_mean"
]].head(10))
print("Rows before and after transformation:", len(df_analysis), len(df_transformed))

Rows before and after transformation: 280 280


,title,country_group,imdb_rating,country_mean_rating,rating_minus_country_mean
0,10 Things I Hate About You,usa,7.3,6.829386,0.470614
2,101 Dalmatians,usa_other,5.7,6.419608,-0.719608
4,102 Dalmatians,usa_other,4.9,6.419608,-1.519608
6,"20,000 Leagues Under the Sea",usa,7.2,6.829386,0.370614
7,A Bug's Life,usa,7.2,6.829386,0.370614
9,A Goofy Movie,usa_other,6.8,6.419608,0.380392
14,A Wrinkle in Time,usa,4.2,6.829386,-2.629386
15,Aladdin,usa,7.0,6.829386,0.170614
16,Aladdin,usa,8.0,6.829386,1.170614
17,Aladdin and the King of Thieves,usa,6.4,6.829386,-0.429386


## 7. Export cleaned data and analysis tables
Save the full cleaned catalogue separately from all movies and the vote-filtered movie sample.
A dictionary and loop apply the same export operation to several outputs.
Annual files contain all cleaned movies from each year, including an `unknown` file if needed.

In [16]:
clean_exports = {
    "disney_shows_clean.csv": df_clean,
    "disney_movies_all.csv": df_movies,
    "disney_movies_clean.csv": df_analysis,
}
for filename, table in clean_exports.items():
    table.to_csv(OUT_CLEAN / filename, index=False)

summary_exports = {
    "movie_ratings_long.csv": df_ratings_long,
    "movie_ratings_wide.csv": df_ratings_back,
    "ratings_by_country_year.csv": df_summary,
    "ratings_by_country_year_wide.csv": df_summary_wide.reset_index(),
    "movies_with_group_ratings.csv": df_transformed,
}
for filename, table in summary_exports.items():
    table.to_csv(OUT_TABLES / filename, index=False)

annual_count = 0
for year, table in df_movies.groupby("year", dropna=False):
    year_label = "unknown" if pd.isna(year) else str(int(year))
    table.to_csv(OUT_ANNUAL / f"disney_movies_clean_{year_label}.csv", index=False)
    annual_count += 1

print("Cleaned datasets:", len(clean_exports))
print("Analysis tables:", len(summary_exports))
print("Annual movie files:", annual_count)
print("Saved under:", OUT)

Cleaned datasets: 3
Analysis tables: 5
Annual movie files: 89
Saved under: /Users/justinasgrigaitis/Documents/Github/ba-programming-2026/sessions/week_4/output
